[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-wholesale.ipynb)

# Full Project: Wholesale Customer Segmentation (Retail)

*AIBits Academy · Machine Learning End To End · Full Project*

440 wholesale-distributor clients, Tukey-IQR outlier removal, PCA, and a Gaussian Mixture Model that rediscovers a hidden business label it was never shown.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

def fetch(url, target, member=None):   # public source; a zip member is extracted and renamed to `target`
    if os.path.exists(target):
        return
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    blob = urllib.request.urlopen(req, timeout=120).read()
    if member:
        blob = zipfile.ZipFile(io.BytesIO(blob)).read(member)
    open(target, 'wb').write(blob)
    print('downloaded', target)

fetch('https://raw.githubusercontent.com/udacity/MLND_CN_P3_Customer_Segments/master/customers.csv', 'customers.csv')

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

**Load the wholesale-customer data** (440 clients, six annual-spend columns). `Channel` and `Region` are set aside; `Channel` is used only at the end to sanity-check the segments.

In [ ]:
raw = pd.read_csv('customers.csv')
data = raw.drop(['Channel', 'Region'], axis=1)
print(data.shape)
data.head()

> **Business Problem**
>
> A wholesale distributor — the same problem faced by cash-and-carry chains like Metro or regional FMCG distributors serving India's kirana-store network — wants to understand what natural customer segments exist in its annual spending data, so delivery schedules, credit terms, and account-management effort can be tailored per segment instead of applied uniformly to every client.

> **Dataset**
>
> **440 clients of a wholesale distributor (Lisbon, Portugal), 6 annual spend categories** — Fresh, Milk, Grocery, Frozen, Detergents_Paper, Delicatessen — plus a `Channel` label (Horeca vs. Retail) withheld from clustering and used only afterward to validate the discovered segments. [Dataset source →](https://raw.githubusercontent.com/udacity/MLND_CN_P3_Customer_Segments/master/customers.csv)

## Step 1 — Are Any Spend Categories Redundant?

Before clustering, a useful check: can each spending category be predicted from the other five? A category that's easily predicted from the rest adds little new information to the segmentation:

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

for feature in data.columns:
    new_data = data.drop([feature], axis=1)
    X_train, X_test, y_train, y_test = train_test_split(new_data, data[feature], test_size=0.25, random_state=45)
    reg = RandomForestRegressor(random_state=45, n_estimators=200).fit(X_train, y_train)
    print(f"{feature}: R^2={r2_score(y_test, reg.predict(X_test)):.4f}")

**Grocery (R²=0.90)** and **Detergents_Paper (R²=0.80)** are both highly predictable from the other categories — clients who spend heavily on one tend to spend heavily on the other, so they carry substantial redundant information. **Fresh, Frozen, and Delicatessen** (R² near zero or negative, meaning a model does *worse* than just predicting the mean) are each contributing genuinely distinct signal that the other categories can't substitute for — exactly the kind of feature-relevance check the Feature Engineering page recommends before deciding what to keep.

## Step 2 — Log-Transform and Remove Multi-Feature Outliers

Annual spend is heavily right-skewed (a few clients spend 10-100x the median) — a log-transform makes the distribution far more clustering-friendly, then Tukey's 1.5×IQR rule flags outliers per feature:

In [ ]:
log_data = np.log(data)
all_outliers = []
for feature in log_data.columns:
    Q1, Q3 = np.percentile(log_data[feature], [25,75])
    step = 1.5 * (Q3 - Q1)
    outliers = log_data[~((log_data[feature]>=Q1-step)&(log_data[feature]<=Q3+step))].index.tolist()
    all_outliers.extend(outliers)

from collections import Counter
counts = Counter(all_outliers)
multi_feature_outliers = [i for i,c in counts.items() if c>1]  # flagged on >1 feature

Only points flagged as outliers on **more than one** feature are removed (5 of 440, just over 1%) — a deliberately conservative rule, since a client being an outlier on one single spending category is normal business variation, not necessarily a data problem, while being an extreme outlier across multiple categories simultaneously is a much stronger signal of a genuinely atypical (or erroneous) record.

## Step 3 — PCA: Two Components Capture 70.7% of Variance

The five points flagged on more than one feature are removed, leaving the cleaned log-scaled data used for PCA.

In [ ]:
good_data = data.drop(data.index[multi_feature_outliers]).reset_index(drop=True)
good_log_data = np.log(good_data)
print(good_data.shape)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=45)
reduced_data = pca.fit_transform(good_log_data)
print("Explained variance:", pca.explained_variance_ratio_, "sum=", pca.explained_variance_ratio_.sum())

Dimension 1 is dominated by **Detergents_Paper and Grocery** (positive loadings around 0.4–0.75) — a "planned bulk restocking" axis. Dimension 2 is dominated by **Fresh and Frozen** (loadings 0.5–0.69) — a "perishables" axis. Two components already capture 70.7% of total variance, enough to cluster on directly and to visualise on a single 2D scatter plot.

## Step 4 — Gaussian Mixture Model: Silhouette Picks k=2

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

for k in range(2,7):
    gmm = GaussianMixture(n_components=k, random_state=45, n_init=5)
    labels = gmm.fit_predict(reduced_data)
    print(f"k={k}: silhouette={silhouette_score(reduced_data, labels):.4f}")

Silhouette peaks clearly at **k=2** (0.4219), dropping steadily as k grows — unlike the Customer Segmentation full project elsewhere in this course (a different dataset, K-Means, elbow-vs-silhouette disagreement favouring k=3), here both the shape of the data and the model-selection metric agree cleanly on two segments.

## Step 5 — Recovering Segment Profiles, and a Validation Surprise

The lesson's final model is the two-component mixture (best silhouette in the sweep above).

In [ ]:
from sklearn.mixture import GaussianMixture
gmm_final = GaussianMixture(n_components=2, random_state=45, n_init=5).fit(reduced_data)

In [ ]:
centers_log = gmm_final.means_
centers_original = np.exp(pca.inverse_transform(centers_log))
print(pd.DataFrame(np.round(centers_original), columns=data.columns))

**Segment 0** (281 clients) spends heavily on Fresh and Frozen but little on Grocery/Detergents_Paper — the profile of a restaurant or cafe buying perishables daily. **Segment 1** (154 clients) spends heavily on Grocery and Detergents_Paper but comparatively little on Fresh — the profile of a retail shop stocking packaged, shelf-stable goods.

The dataset happens to include a `Channel` label (Horeca = hotel/restaurant/cafe, vs. Retail) that was **deliberately withheld from clustering**. Cross-tabulating the discovered segments against it afterward:

Without ever seeing the Channel label, unsupervised clustering placed **266 of 294 Horeca clients (90.5%)** into Segment 0 and **126 of 141 Retail clients (89.4%)** into Segment 1 — the purely spending-pattern-based segmentation recovered a real, independently-meaningful business distinction almost perfectly. This is the single most convincing evidence that these two discovered segments reflect a genuine underlying business reality, not an arbitrary split.

## Visualizing the Recovery

Green cells are where the unsupervised segment matches the true Channel label — both diagonal cells dominate their row, with gold rings marking the two large "recovered correctly" counts.

> **💡 Why Validate Against a Label You Didn't Cluster On**
>
> This mirrors the "stepping back: solving a business problem vs. data exploration" caution that applies to any unsupervised result: clustering will always produce *some* segments, whether or not they mean anything. Having an independent, business-meaningful label to check against — even one excluded from the clustering itself — is one of the strongest ways to build confidence that discovered segments are real rather than an artifact of whichever k and algorithm happened to be tried.

## Key Business Takeaways

- Grocery and Detergents_Paper spend are highly redundant with the other categories (R²=0.90 and 0.80) — Fresh, Frozen, and Delicatessen carry the genuinely distinct signal driving segmentation.
- Only 5 of 440 clients (1.1%) were removed as multi-feature outliers — a deliberately conservative outlier rule that avoids discarding legitimate business variation.
- The Gaussian Mixture Model's 2-segment split, discovered from spending patterns alone, recovered the real Horeca-vs-Retail channel distinction with ~90% agreement on both sides — strong evidence the segments are business-meaningful, not arbitrary.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Skew before and after the log

Store the skewness of the raw `Milk` column in `skew_raw` and of `np.log(Milk)` in `skew_log`.

In [ ]:
skew_raw = skew_log = None   # TODO


In [ ]:
try:
    check("raw is strongly skewed", skew_raw > 2)
    check("log is far more symmetric", abs(skew_log) < 0.5)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
skew_raw = float(data["Milk"].skew())
skew_log = float(np.log(data["Milk"]).skew())

```

</details>

### Exercise 2 · Medium · Variance explained by two components

Store the total variance explained by the two principal components of `good_log_data` in `explained` (about 0.707).

In [ ]:
explained = None   # TODO


In [ ]:
try:
    check("about 70.7%", abs(explained - 0.7068) < 0.002)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
explained = float(pca.explained_variance_ratio_.sum())

```

</details>

### Exercise 3 · Stretch · Do the segments match the true channel?

Predict the segment of every kept customer with `gmm_final.predict(reduced_data)`. Cross-tabulate it against the true `Channel` (from `raw`, same rows as `good_data`) and store the share of customers where the majority mapping agrees in `agreement`. It should be high (> 0.85).

In [ ]:
agreement = None   # TODO


In [ ]:
try:
    check("segments line up with the sales channel", agreement > 0.85)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
seg = gmm_final.predict(reduced_data)
channel = raw.drop(raw.index[multi_feature_outliers]).reset_index(drop=True)["Channel"]
ct = pd.crosstab(seg, channel)
agreement = float(ct.max(axis=1).sum() / ct.values.sum())

```

Clustering never saw `Channel`, yet the two discovered segments line up with it - evidence that the structure is real.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Wholesale Customer Segmentation (Retail)**.*